In [1]:
!pip install -U transformers datasets accelerate sentencepiece -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 84.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.3 MB/s eta 0:00:00


In [14]:
import torch
import numpy as np
import pandas as pd
import json

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer
)

from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

NameError: name 'tokenizer' is not defined

In [4]:
datasets = {
    "Barishal": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Barishal_NER.csv"),
    "Chittagong": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Chittagong_NER.csv"),
    "Mymensingh": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Mymensingh_NER.csv"),
    "Noakhali": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Noakhali_NER.csv"),
    "Sylhet": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Sylhet_NER.csv")
}

print("Datasets loaded.")

Datasets loaded.


In [18]:
from transformers import AutoTokenizer

MODEL_NAME = "csebuetnlp/banglabert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded.")

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded.


In [19]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

print("Data collator ready.")

Data collator ready.


In [20]:
sentence_data = {}

for dialect, df in datasets.items():

    df["Sentence #"] = df["Sentence #"].ffill()

    word_column = [
        c for c in df.columns
        if c.endswith("_word")
    ][0]

    sentences = []

    for sentence_id, group in df.groupby("Sentence #"):

        sentences.append({
            "sentence_id": int(
                str(sentence_id).replace("Sentence: ", "")
            ),
            "tokens": group[word_column].astype(str).tolist(),
            "tags": group["bio_tag"].astype(str).tolist()
        })

    sentence_data[dialect] = sentences

print("Sentence conversion complete.")

for dialect in sentence_data:
    print(dialect, len(sentence_data[dialect]))

Sentence conversion complete.
Barishal 3481
Chittagong 3481
Mymensingh 3481
Noakhali 3481
Sylhet 3481


In [21]:
with open("/kaggle/input/datasets/arnav1605/ancholink-ner/within_dialect_splits.json","r") as f:
    splits = json.load(f)

print("Splits loaded.")

Splits loaded.


In [22]:
for dialect, split in splits.items():
    print(
        dialect,
        "| Train:", len(split["train"]),
        "| Test:", len(split["test"])
    )

Barishal | Train: 2784 | Test: 697
Chittagong | Train: 2784 | Test: 697
Mymensingh | Train: 2784 | Test: 697
Noakhali | Train: 2784 | Test: 697
Sylhet | Train: 2784 | Test: 697


In [23]:
tags = set()

for sentences in sentence_data.values():

    for sentence in sentences:
        tags.update(sentence["tags"])

tags = sorted(tags)

tag_to_id = {
    tag: i
    for i, tag in enumerate(tags)
}

id_to_tag = {
    i: tag
    for tag, i in tag_to_id.items()
}

print("Number of BIO tags:", len(tag_to_id))
print(tag_to_id)

Number of BIO tags: 20
{'\nI-LOC': 0, 'B-ANI': 1, 'B-COL': 2, 'B-FOOD': 3, 'B-LOC': 4, 'B-OBJ': 5, 'B-ORG': 6, 'B-PER': 7, 'B-REL': 8, 'B-ROLE': 9, 'I-ANI': 10, 'I-COL': 11, 'I-FOOD': 12, 'I-LOC': 13, 'I-OBJ': 14, 'I-ORG': 15, 'I-PER': 16, 'I-REL': 17, 'I-ROLE': 18, 'O': 19}


In [9]:
!pip install git+https://github.com/csebuetnlp/normalizer -q

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 5.0 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [24]:
dialect = "Barishal"

train_ids = set(splits[dialect]["train"])
test_ids = set(splits[dialect]["test"])

train_sentences = [
    s for s in sentence_data[dialect]
    if s["sentence_id"] in train_ids
]

test_sentences = [
    s for s in sentence_data[dialect]
    if s["sentence_id"] in test_ids
]

print("Train sentences:", len(train_sentences))
print("Test sentences:", len(test_sentences))

Train sentences: 2784
Test sentences: 697


In [25]:
sentence = train_sentences[0]

text = " ".join(sentence["tokens"])

print("Original tokens:")
print(sentence["tokens"])

print("\nOriginal tags:")
print(sentence["tags"])

print("\nOriginal sentence:")
print(text)

Original tokens:
['মাইআউগ্গা', 'হেয়ার', 'শব', 'হপ্পনের', 'মইদ্ধে', 'পলাউগ্গারে', 'ভাব্তে', 'থাহে']

Original tags:
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Original sentence:
মাইআউগ্গা হেয়ার শব হপ্পনের মইদ্ধে পলাউগ্গারে ভাব্তে থাহে


In [26]:
from normalizer import normalize

normalized_text = normalize(text)

print("Normalized sentence:")
print(normalized_text)

Normalized sentence:
মাইআউগ্গা হেয়ার শব হপ্পনের মইদ্ধে পলাউগ্গারে ভাব্তে থাহে


In [27]:
tokens = tokenizer.tokenize(normalized_text)

print(tokens)

['মাই', '##আ', '##উ', '##গ', '##্গা', 'হেয়ার', 'শব', 'হ', '##প্প', '##নের', 'মই', '##দ্ধে', 'পলা', '##উ', '##গ', '##্গা', '##রে', 'ভাব', '##্তে', 'থা', '##হে']


In [28]:
encoded = tokenizer(
    sentence["tokens"],
    is_split_into_words=True,
    truncation=True
)

word_ids = encoded.word_ids()

print("Subwords:")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

print("\nWord IDs:")
print(word_ids)

Subwords:
['[CLS]', 'মাই', '##আ', '##উ', '##গ', '##্গা', 'হেয়ার', 'শব', 'হ', '##প্প', '##নের', 'মই', '##দ্ধে', 'পলা', '##উ', '##গ', '##্গা', '##রে', 'ভাব', '##্তে', 'থা', '##হে', '[SEP]']

Word IDs:
[None, 0, 0, 0, 0, 0, 1, 2, 3, 3, 3, 4, 4, 5, 5, 5, 5, 5, 6, 6, 7, 7, None]


In [29]:
labels = []

previous_word_id = None

for word_id in word_ids:

    # Special tokens such as [CLS] and [SEP]
    if word_id is None:
        labels.append(-100)

    # First subword of a word
    elif word_id != previous_word_id:

        labels.append(
            tag_to_id[sentence["tags"][word_id]]
        )

    # Additional subword
    else:

        original_tag = sentence["tags"][word_id]

        if original_tag.startswith("B-"):
            original_tag = "I-" + original_tag[2:]

        labels.append(
            tag_to_id[original_tag]
        )

    previous_word_id = word_id

print("Subwords:")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

print("\nAligned labels:")
print([
    -100 if x == -100 else id_to_tag[x]
    for x in labels
])

Subwords:
['[CLS]', 'মাই', '##আ', '##উ', '##গ', '##্গা', 'হেয়ার', 'শব', 'হ', '##প্প', '##নের', 'মই', '##দ্ধে', 'পলা', '##উ', '##গ', '##্গা', '##রে', 'ভাব', '##্তে', 'থা', '##হে', '[SEP]']

Aligned labels:
[-100, 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', -100]


In [30]:
def tokenize_and_align(sentence):

    encoded = tokenizer(
        sentence["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=512
    )

    word_ids = encoded.word_ids()

    labels = []
    previous_word_id = None

    for word_id in word_ids:

        if word_id is None:
            labels.append(-100)

        elif word_id != previous_word_id:

            labels.append(
                tag_to_id[sentence["tags"][word_id]]
            )

        else:

            tag = sentence["tags"][word_id]

            if tag.startswith("B-"):
                tag = "I-" + tag[2:]

            labels.append(tag_to_id[tag])

        previous_word_id = word_id

    encoded["labels"] = labels

    return encoded

In [31]:
from datasets import Dataset

tokenized_data = {}

for dialect in sentence_data:

    train_ids = set(splits[dialect]["train"])
    test_ids = set(splits[dialect]["test"])

    train_sentences = [
        s for s in sentence_data[dialect]
        if s["sentence_id"] in train_ids
    ]

    test_sentences = [
        s for s in sentence_data[dialect]
        if s["sentence_id"] in test_ids
    ]

    # Apply the same preprocessing function to every sentence
    train_encoded = [
        tokenize_and_align(s)
        for s in train_sentences
    ]

    test_encoded = [
        tokenize_and_align(s)
        for s in test_sentences
    ]

    tokenized_data[dialect] = {
        "train": Dataset.from_list(train_encoded),
        "test": Dataset.from_list(test_encoded)
    }

    print(
        dialect,
        "| Train:", len(train_encoded),
        "| Test:", len(test_encoded)
    )

Barishal | Train: 2784 | Test: 697
Chittagong | Train: 2784 | Test: 697
Mymensingh | Train: 2784 | Test: 697
Noakhali | Train: 2784 | Test: 697
Sylhet | Train: 2784 | Test: 697


In [32]:
for dialect, data in tokenized_data.items():

    print(
        dialect,
        "| Train:", len(data["train"]),
        "| Test:", len(data["test"])
    )

    assert len(data["train"]) == 2784
    assert len(data["test"]) == 697

Barishal | Train: 2784 | Test: 697
Chittagong | Train: 2784 | Test: 697
Mymensingh | Train: 2784 | Test: 697
Noakhali | Train: 2784 | Test: 697
Sylhet | Train: 2784 | Test: 697


In [33]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(tag_to_id),
    id2label=id_to_tag,
    label2id=tag_to_id
)

print("Model loaded.")

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForTokenClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.weight                                 | MISSING    | 
classifier.bias                                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.


In [34]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./banglabert_barishal",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="no",
    report_to="none"
)

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

In [35]:
def get_entities(labels):
    entities = []
    start = None
    entity_type = None

    for i, tag in enumerate(labels + ["O"]):

        if tag.startswith("B-"):
            if start is not None:
                entities.append((start, i - 1, entity_type))

            start = i
            entity_type = tag[2:]

        elif tag.startswith("I-"):
            if start is None:
                start = i
                entity_type = tag[2:]

        else:
            if start is not None:
                entities.append((start, i - 1, entity_type))

            start = None
            entity_type = None

    return set(entities)


def calculate_metrics(predictions, labels):

    tp = 0
    predicted = 0
    actual = 0

    for pred, true in zip(predictions, labels):

        pred_entities = get_entities(pred)
        true_entities = get_entities(true)

        tp += len(pred_entities & true_entities)
        predicted += len(pred_entities)
        actual += len(true_entities)

    precision = tp / predicted if predicted else 0
    recall = tp / actual if actual else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [36]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    all_predictions = []
    all_labels = []

    for pred, label in zip(predictions, labels):

        pred_tags = []
        label_tags = []

        for p, l in zip(pred, label):

            if l == -100:
                continue

            pred_tags.append(id_to_tag[p])
            label_tags.append(id_to_tag[l])

        all_predictions.append(pred_tags)
        all_labels.append(label_tags)

    return calculate_metrics(all_predictions, all_labels)

In [37]:
results = {}

for dialect in tokenized_data:

    print("\n" + "=" * 50)
    print("Training:", dialect)
    print("=" * 50)

    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(tag_to_id),
        id2label=id_to_tag,
        label2id=tag_to_id
    )

    training_args = TrainingArguments(
        output_dir=f"/kaggle/working/results_{dialect}",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=20,
        weight_decay=0.01,
        logging_steps=100,
        save_strategy="no",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_data[dialect]["train"],
        eval_dataset=tokenized_data[dialect]["test"],
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    metrics = trainer.evaluate()

    results[dialect] = metrics

    print(metrics)


Training: Barishal


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForTokenClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.weight                                 | MISSING    | 
classifier.bias                                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input t

Step,Training Loss
100,1.768692
200,0.846479
300,0.671580
400,0.498198
500,0.420540
600,0.327330
700,0.290939
800,0.239454
900,0.197307
1000,0.176215


Training Loss,Validation Loss,Step,Precision,Recall,F1
0.023454,0.421699,3480,0.771208,0.753769,0.762389


{'eval_loss': 0.42169877886772156, 'eval_precision': 0.7712082262210797, 'eval_recall': 0.7537688442211056, 'eval_f1': 0.7623888182973316}

Training: Chittagong


... (16 outputs left out)

[transformers] ElectraForTokenClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.weight                                 | MISSING    | 
classifier.bias                                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input t

Step,Training Loss
100,1.417162
200,0.758981
300,0.591809
400,0.427129
500,0.351187
600,0.278111
700,0.251680
800,0.190653
900,0.190125
1000,0.139304


Training Loss,Validation Loss,Step,Precision,Recall,F1
0.022604,0.435567,3480,0.781638,0.789474,0.785536


{'eval_loss': 0.43556737899780273, 'eval_precision': 0.7816377171215881, 'eval_recall': 0.7894736842105263, 'eval_f1': 0.7855361596009975}


In [38]:
import pandas as pd

results_table = pd.DataFrame([
    {
        "Dialect": dialect,
        "Precision": results[dialect]["eval_precision"],
        "Recall": results[dialect]["eval_recall"],
        "F1": results[dialect]["eval_f1"]
    }
    for dialect in results
])

results_table

,Dialect,Precision,Recall,F1
0,Barishal,0.771208,0.753769,0.762389
1,Chittagong,0.754569,0.741026,0.747736
2,Mymensingh,0.809045,0.815190,0.812106
3,Noakhali,0.782051,0.772152,0.777070
4,Sylhet,0.781638,0.789474,0.785536
